In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
import os
warnings.filterwarnings('ignore')

In [17]:
# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [18]:
print("\n1. CHARGEMENT DES DONNÉES")

# Modifier ce chemin selon votre structure
df = pd.read_csv('../data/raw/dataset.csv')

print(f"✓ Dataset chargé avec succès")
print(f"  Nombre total de lignes : {len(df):,}")
print(f"  Nombre de colonnes : {len(df.columns)}")


1. CHARGEMENT DES DONNÉES
✓ Dataset chargé avec succès
  Nombre total de lignes : 20,000
  Nombre de colonnes : 15


In [19]:
print("\n2. STRUCTURE DU DATASET")
print("-"*80)

print("\nColonnes disponibles :")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nPremières lignes du dataset :")
print(df.head())

print("\nTypes de données :")
print(df.dtypes)

print("\nInformations générales :")
df.info()


2. STRUCTURE DU DATASET
--------------------------------------------------------------------------------

Colonnes disponibles :
   1. subject
   2. body
   3. answer
   4. type
   5. queue
   6. priority
   7. language
   8. tag_1
   9. tag_2
  10. tag_3
  11. tag_4
  12. tag_5
  13. tag_6
  14. tag_7
  15. tag_8

Premières lignes du dataset :
                                             subject  \
0  Unvorhergesehener Absturz der Datenanalyse-Pla...   
1                           Customer Support Inquiry   
2                      Data Analytics for Investment   
3                 Krankenhaus-Dienstleistung-Problem   
4                                           Security   

                                                body  \
0  Die Datenanalyse-Plattform brach unerwartet ab...   
1  Seeking information on digital strategies that...   
2  I am contacting you to request information on ...   
3  Ein Medien-Daten-Sperrverhalten trat aufgrund ...   
4  Dear Customer Support, I am reac

In [20]:
print("\n3. ANALYSE DES VALEURS MANQUANTES")
print("-"*80)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing.index,
    'Valeurs manquantes': missing.values,
    'Pourcentage (%)': missing_pct.values
})
missing_df = missing_df[missing_df['Valeurs manquantes'] > 0].sort_values('Valeurs manquantes', ascending=False)

if len(missing_df) > 0:
    print("\nColonnes avec valeurs manquantes :")
    print(missing_df.to_string(index=False))
else:
    print("✓ Aucune valeur manquante détectée")

# Vérification des colonnes essentielles
essential_cols = ['subject', 'body', 'type']
print("\nVérification des colonnes essentielles :")
for col in essential_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        print(f"  ⚠ {col}: {missing_count} valeurs manquantes ({missing_count/len(df)*100:.2f}%)")
    else:
        print(f"  ✓ {col}: Aucune valeur manquante")


3. ANALYSE DES VALEURS MANQUANTES
--------------------------------------------------------------------------------

Colonnes avec valeurs manquantes :
Colonne  Valeurs manquantes  Pourcentage (%)
  tag_8               18093           90.465
  tag_7               16072           80.360
  tag_6               12649           63.245
  tag_5                6909           34.545
  tag_4                1539            7.695
subject                1461            7.305
  tag_3                  95            0.475
  tag_2                  46            0.230
 answer                   4            0.020
   body                   2            0.010

Vérification des colonnes essentielles :
  ⚠ subject: 1461 valeurs manquantes (7.31%)
  ⚠ body: 2 valeurs manquantes (0.01%)
  ✓ type: Aucune valeur manquante


In [21]:
print("\n4. DISTRIBUTION DES TYPES DE TICKETS")
print("-"*80)

type_counts = df['type'].value_counts()
print("\nNombre de tickets par type :")
print(type_counts)

print(f"\n✓ Nombre de classes différentes : {df['type'].nunique()}")
print(f"✓ Type le plus fréquent : {type_counts.index[0]} ({type_counts.values[0]:,} tickets)")
print(f"✓ Type le moins fréquent : {type_counts.index[-1]} ({type_counts.values[-1]:,} tickets)")

# Vérifier le déséquilibre
max_count = type_counts.max()
min_count = type_counts.min()
imbalance_ratio = max_count / min_count
print(f"\n⚠ Ratio de déséquilibre : {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("  → Classes DÉSÉQUILIBRÉES (utiliser class_weight='balanced')")
else:
    print("  → Classes relativement équilibrées")

# Créer le dossier pour les graphiques
os.makedirs('reports/eda', exist_ok=True)

# Graphique de distribution
plt.figure(figsize=(10, 6))
bars = type_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution des Types de Tickets', fontsize=16, fontweight='bold')
plt.xlabel('Type de Ticket', fontsize=12)
plt.ylabel('Nombre de Tickets', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres
for i, v in enumerate(type_counts.values):
    plt.text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/eda/type_distribution.png', dpi=300, bbox_inches='tight')
print("\n✓ Graphique sauvegardé : reports/eda/type_distribution.png")
plt.close()


4. DISTRIBUTION DES TYPES DE TICKETS
--------------------------------------------------------------------------------

Nombre de tickets par type :
type
Incident    7978
Request     5763
Problem     4184
Change      2075
Name: count, dtype: int64

✓ Nombre de classes différentes : 4
✓ Type le plus fréquent : Incident (7,978 tickets)
✓ Type le moins fréquent : Change (2,075 tickets)

⚠ Ratio de déséquilibre : 3.84:1
  → Classes DÉSÉQUILIBRÉES (utiliser class_weight='balanced')

✓ Graphique sauvegardé : reports/eda/type_distribution.png


In [22]:
print("\n5. STATISTIQUES TEXTUELLES")
print("-"*80)

# Créer une colonne combinée temporaire
df['text_combined_temp'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')

# Longueur des textes
df['text_length'] = df['text_combined_temp'].str.len()
df['word_count'] = df['text_combined_temp'].str.split().str.len()

print("\nLongueurs des emails (subject + body) :")
print(f"  Longueur moyenne : {df['text_length'].mean():.0f} caractères")
print(f"  Longueur médiane : {df['text_length'].median():.0f} caractères")
print(f"  Longueur minimale : {df['text_length'].min():.0f} caractères")
print(f"  Longueur maximale : {df['text_length'].max():.0f} caractères")
print(f"  Écart-type : {df['text_length'].std():.0f} caractères")

print("\nNombre de mots :")
print(f"  Moyenne : {df['word_count'].mean():.1f} mots")
print(f"  Médiane : {df['word_count'].median():.0f} mots")
print(f"  Min : {df['word_count'].min():.0f} mots")
print(f"  Max : {df['word_count'].max():.0f} mots")

# Textes très courts
short_texts = df[df['word_count'] < 5]
print(f"\n⚠ Tickets avec < 5 mots : {len(short_texts)} ({len(short_texts)/len(df)*100:.2f}%)")

# Distribution des longueurs
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df['text_length'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution des Longueurs de Texte', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nombre de caractères', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Moyenne: {df["text_length"].mean():.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(df['word_count'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_title('Distribution du Nombre de Mots', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nombre de mots', fontsize=11)
axes[1].set_ylabel('Fréquence', fontsize=11)
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Moyenne: {df["word_count"].mean():.1f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/eda/text_length_distribution.png', dpi=300, bbox_inches='tight')
print("\n✓ Graphique sauvegardé : reports/eda/text_length_distribution.png")
plt.close()



5. STATISTIQUES TEXTUELLES
--------------------------------------------------------------------------------

Longueurs des emails (subject + body) :
  Longueur moyenne : 437 caractères
  Longueur médiane : 401 caractères
  Longueur minimale : 5 caractères
  Longueur maximale : 2284 caractères
  Écart-type : 250 caractères

Nombre de mots :
  Moyenne : 62.2 mots
  Médiane : 57 mots
  Min : 1 mots
  Max : 283 mots

⚠ Tickets avec < 5 mots : 12 (0.06%)

✓ Graphique sauvegardé : reports/eda/text_length_distribution.png


In [23]:
print("\n6. ANALYSE PAR LANGUE")
print("-"*80)

if 'language' in df.columns:
    lang_counts = df['language'].value_counts()
    print("\nDistribution des langues :")
    print(lang_counts)
    print()
    for lang, count in lang_counts.items():
        pct = (count / len(df)) * 100
        print(f"  {lang}: {count:,} tickets ({pct:.1f}%)")



6. ANALYSE PAR LANGUE
--------------------------------------------------------------------------------

Distribution des langues :
language
en    11923
de     8077
Name: count, dtype: int64

  en: 11,923 tickets (59.6%)
  de: 8,077 tickets (40.4%)


In [24]:
print("\n7. ANALYSE DES PRIORITÉS")
print("-"*80)

if 'priority' in df.columns:
    priority_counts = df['priority'].value_counts()
    print("\nDistribution des priorités :")
    print(priority_counts)


7. ANALYSE DES PRIORITÉS
--------------------------------------------------------------------------------

Distribution des priorités :
priority
medium    8144
high      7801
low       4055
Name: count, dtype: int64


In [25]:
print("\n8. LONGUEUR MOYENNE PAR TYPE DE TICKET")
print("-"*80)

avg_length_by_type = df.groupby('type')['word_count'].agg(['mean', 'median', 'std'])
avg_length_by_type = avg_length_by_type.sort_values('mean', ascending=False)
print("\nStatistiques par type :")
print(avg_length_by_type.round(1))

# Boxplot
plt.figure(figsize=(12, 6))
df.boxplot(column='word_count', by='type', patch_artist=True)
plt.suptitle('')  # Supprimer le titre automatique
plt.title('Distribution du Nombre de Mots par Type de Ticket', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Type de Ticket', fontsize=12)
plt.ylabel('Nombre de mots', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/eda/wordcount_by_type.png', dpi=300, bbox_inches='tight')
print("\n✓ Graphique sauvegardé : reports/eda/wordcount_by_type.png")
plt.close()


8. LONGUEUR MOYENNE PAR TYPE DE TICKET
--------------------------------------------------------------------------------

Statistiques par type :
          mean  median   std
type                        
Change    67.8    69.0  39.4
Request   64.4    61.0  37.6
Incident  60.1    53.0  35.4
Problem   60.1    52.0  36.6

✓ Graphique sauvegardé : reports/eda/wordcount_by_type.png


<Figure size 1200x600 with 0 Axes>

In [26]:
print("\n9. APERÇU DES MOTS FRÉQUENTS")
print("-"*80)

# Échantillon pour avoir une idée
all_words = ' '.join(df['text_combined_temp'].head(1000)).lower().split()
word_counts = Counter(all_words)
most_common = word_counts.most_common(20)

print("\n20 mots les plus fréquents (échantillon de 1000 tickets) :")
for word, count in most_common:
    print(f"  {word:20s} : {count:5d}")

print("\n⚠ Note : Ces mots incluent les stopwords. Le preprocessing les supprimera.")



9. APERÇU DES MOTS FRÉQUENTS
--------------------------------------------------------------------------------

20 mots les plus fréquents (échantillon de 1000 tickets) :
  the                  :  1867
  to                   :  1772
  and                  :  1471
  i                    :   925
  you                  :   854
  for                  :   839
  data                 :   673
  in                   :   639
  with                 :   565
  your                 :   562
  on                   :   521
  a                    :   506
  und                  :   502
  die                  :   462
  zu                   :   438
  this                 :   437
  of                   :   435
  am                   :   421
  could                :   421
  problem              :   395

⚠ Note : Ces mots incluent les stopwords. Le preprocessing les supprimera.


In [27]:
print("\n10. EXEMPLES DE TICKETS")
print("-"*80)

for i in range(min(3, len(df))):
    print(f"\n[Exemple {i+1}]")
    print(f"Type: {df.iloc[i]['type']}")
    print(f"Langue: {df.iloc[i]['language']}")
    print(f"Priorité: {df.iloc[i]['priority']}")
    print(f"Subject: {df.iloc[i]['subject']}")
    print(f"Body (extrait): {str(df.iloc[i]['body'])[:150]}...")
    print(f"Nombre de mots: {df.iloc[i]['word_count']}")
    print("-"*80)



10. EXEMPLES DE TICKETS
--------------------------------------------------------------------------------

[Exemple 1]
Type: Incident
Langue: de
Priorité: low
Subject: Unvorhergesehener Absturz der Datenanalyse-Plattform
Body (extrait): Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe versucht, Laravel 8 und meinen MacBook Pro neu zu st...
Nombre de mots: 42
--------------------------------------------------------------------------------

[Exemple 2]
Type: Request
Langue: en
Priorité: medium
Subject: Customer Support Inquiry
Body (extrait): Seeking information on digital strategies that can aid in brand growth and details on the available services. Looking forward to learning more to help...
Nombre de mots: 41
--------------------------------------------------------------------------------

[Exemple 3]
Type: Request
Langue: en
Priorité: medium
Subject: Data Analytics for Investment
Body (extrait): I am contacting you to request informati

In [ ]:
print("\n" + "="*80)
print("RÉSUMÉ ET RECOMMANDATIONS")
print("="*80)

print(f"\n✓ RÉSUMÉ :")
print(f"  • Dataset : {len(df):,} tickets")
print(f"  • Classes : {df['type'].nunique()} types différents")
print(f"  • Langues : {df['language'].nunique() if 'language' in df.columns else 'N/A'}")
print(f"  • Longueur moyenne : {df['word_count'].mean():.1f} mots")
print(f"  • Textes courts (<5 mots) : {len(short_texts)} tickets")

print(f"\n✓ DISTRIBUTION DES CLASSES :")
for ticket_type, count in df['type'].value_counts().items():
    pct = (count / len(df)) * 100
    print(f"  • {ticket_type:12s} : {count:5,} tickets ({pct:5.1f}%)")

print(f"\n⚠ POINTS D'ATTENTION :")
if imbalance_ratio > 3:
    print(f"  • Classes déséquilibrées (ratio {imbalance_ratio:.2f}:1)")
    print(f"    → Utiliser class_weight='balanced'")
print(f"  • Dataset multilingue (EN + DE)")
print(f"    → Utiliser un modèle d'embedding multilingue")
if df['subject'].isnull().sum() > 0:
    print(f"  • {df['subject'].isnull().sum()} valeurs manquantes dans 'subject'")

print(f"\nRECOMMANDATIONS PREPROCESSING :")
print(f"  1. Fusionner subject + body")
print(f"  2. Nettoyage : lowercase, ponctuation, URLs, emails")
print(f"  3. Suppression stopwords EN + DE")
print(f"  4. Tokenisation")
print(f"  5. Normalisation")

print(f"\n RECOMMANDATIONS MODÉLISATION :")
print(f"  1. Modèle embedding : paraphrase-multilingual-MiniLM-L12-v2")
print(f"  2. Classification : class_weight='balanced'")
print(f"  3. Validation : split stratifié 80/20")
print(f"  4. Métriques : F1-Score, Precision, Recall par classe")

print("\n" + "="*80)
print("ANALYSE TERMINÉE ✓")
print("="*80)
print("\nFichiers générés :")
print("  • reports/eda/type_distribution.png")
print("  • reports/eda/text_length_distribution.png")
print("  • reports/eda/wordcount_by_type.png")

# Nettoyer les colonnes temporaires
df = df.drop(['text_combined_temp', 'text_length', 'word_count'], axis=1)

print("\n✓ Prêt pour l'étape suivante : Preprocessing NLP")


RÉSUMÉ ET RECOMMANDATIONS

✓ RÉSUMÉ :
  • Dataset : 20,000 tickets
  • Classes : 4 types différents
  • Langues : 2
  • Longueur moyenne : 62.2 mots
  • Textes courts (<5 mots) : 12 tickets

✓ DISTRIBUTION DES CLASSES :
  • Incident     : 7,978 tickets ( 39.9%)
  • Request      : 5,763 tickets ( 28.8%)
  • Problem      : 4,184 tickets ( 20.9%)
  • Change       : 2,075 tickets ( 10.4%)

⚠ POINTS D'ATTENTION :
  • Classes déséquilibrées (ratio 3.84:1)
    → Utiliser class_weight='balanced'
  • Dataset multilingue (EN + DE)
    → Utiliser un modèle d'embedding multilingue
  • 1461 valeurs manquantes dans 'subject'

RECOMMANDATIONS PREPROCESSING :
  1. Fusionner subject + body
  2. Nettoyage : lowercase, ponctuation, URLs, emails
  3. Suppression stopwords EN + DE
  4. Tokenisation
  5. Normalisation

RECOMMANDATIONS MODÉLISATION :
  1. Modèle embedding : paraphrase-multilingual-MiniLM-L12-v2
  2. Classification : class_weight='balanced'
  3. Validation : split stratifié 80/20
  4. Métriq